# Scenario Playground Step 6: Scenario Shocks

Objective: recreate the eight user-facing shock concepts on the selected event-aware rows using only cached, point-in-time source data.

Success means every formula, source, unit, event match, proxy disclosure, and unavailable value is explicit. This notebook does not load targets or state features and does not assemble a model matrix.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.shock_features import (
    HISTORICAL_SHOCK_NAMES,
    SHOCK_FEATURE_NAMES,
    build_shock_artifacts,
    persist_shock_artifacts,
)

artifacts = build_shock_artifacts()
output_paths = persist_shock_artifacts(artifacts)
features = artifacts["features"]
event_audit = artifacts["event_audit"]
market_audit = artifacts["market_audit"]
summary = artifacts["summary"]
print({"rows": len(features), "shock_features": len(HISTORICAL_SHOCK_NAMES), "numeric_features_with_flags": len(SHOCK_FEATURE_NAMES)})
output_paths

## Contract mapping and definitions

CPI and GDP remain true actual-minus-expectation surprises. The Kansas City manufacturing series is intentionally retained under an internal regional-proxy name and maps to the UI's `pmi_change_points` input.

In [ ]:
definition_rows = []
for feature, definition in summary["definitions"].items():
    definition_rows.append({"feature": feature, **definition})
definitions = pd.DataFrame(definition_rows)
print("Contract mapping:", summary["contract_to_historical_feature"])
definitions

## Event matching, release flags, and missingness

Macro shocks are zero only when that event type is absent. A released hold or an exactly neutral surprise can also be zero, so the release flags preserve that distinction.

In [ ]:
coverage = pd.DataFrame({
    "events": pd.Series(summary["event_counts"]),
    "release_flags": pd.Series(summary["release_flag_counts"]),
}).fillna(0).astype(int)
missingness = pd.Series(summary["missingness"], name="missing_rows")
print(coverage)
print("\nNonzero missingness:")
print(missingness[missingness > 0])
features.head(3)

## Historical ranges

These are construction diagnostics, not clipping rules. Sparse event shocks include structural zeros on rows for other event types.

In [ ]:
range_table = pd.DataFrame(summary["distributions"]).T[
    ["count", "missing_count", "nonzero_count", "minimum", "median", "maximum", "mean", "standard_deviation"]
]
range_table

## Point-in-time audit

CPI expectations must be strictly before release. All market source dates must be at or before their Step 4 window boundary. The one unavailable July 2013 CPI surprise remains missing rather than using a post-release value.

In [ ]:
cpi_events = event_audit[event_audit["event_type"] == "cpi"].copy()
unavailable = cpi_events[~cpi_events["feature_available"]]
print("Unavailable CPI events:")
print(unavailable[["sample_date", "event_observation_period", "original_release_date", "notes"]].to_string(index=False))
print("\nMarket source staleness:")
print(pd.DataFrame(summary["market_source_staleness"]).T)
assert event_audit["point_in_time_pass"].all()
assert market_audit["point_in_time_pass"].all()
unavailable

## Step 6 acceptance decision

Every non-missing value is reproducible from a named cached source. The single source-boundary CPI gap is explicit and is left for Step 8's missing-row policy; it is not imputed here.

In [ ]:
validation = pd.Series(summary["validations"], name="passed")
print(validation.to_string())
assert validation.all()
assert len(features) == 585
assert features[list(HISTORICAL_SHOCK_NAMES)].isna().sum().sum() == 1
print("STEP_6_COMPLETE_WITH_ONE_DOCUMENTED_SOURCE_BOUNDARY_MISSING_VALUE")